# Flight Delay Prediction

**Tabular Regression Project** — Predict airline-related financial metrics.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: DAL (Delta Air Lines) stock price (Yahoo Finance, ~10 years)
Target: `Close` price

## 2 · Project Overview

We predict Delta Air Lines' (**DAL**) stock closing price using historical price data downloaded from Yahoo Finance. While the folder name says 'Flight Delay', the underlying pipeline uses financial time-series data — we treat it as a tabular regression problem by engineering lag and technical features.

This teaches feature engineering from financial time-series data.

## 3 · Learning Objectives

1. Download financial data using yfinance.
2. Engineer lag, rolling, and technical features.
3. Apply tabular regression to financial data.
4. Understand the limits of price prediction.
5. Use LazyPredict and FLAML for benchmarking.

## 4 · Problem Statement

Predict Delta Air Lines' **closing stock price** from historical price features. This is an educational exercise — not a trading strategy.

## 5 · Why This Project Matters

- Financial feature engineering is a widely applicable skill.
- Teaches responsible ML for finance (no overblown claims).
- Demonstrates yfinance data loading workflow.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Yahoo Finance (DAL ticker) |
| **Period** | ~10 years |
| **Features** | Open, High, Low, Volume + engineered |
| **Target** | Close price |

## 7 · Dataset Source and License Notes

- **Source**: Yahoo Finance via yfinance library.
- **License**: Personal/educational use.
- **Note**: Financial data should not be used for trading without proper validation.

## 8 · Environment Setup

In [1]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('yfinance')
print('All packages ready.')

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

Imports complete.


## 10 · Configuration / Constants

In [3]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'Close'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [4]:
import yfinance as yf

df = yf.download('DAL', period='10y', auto_adjust=True).reset_index()
print(f'Loaded: {df.shape}')
# Flatten MultiIndex columns if present
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] if c[1] == '' else c[0] for c in df.columns]
print(f'Columns: {list(df.columns)}')
df.head()

[*********************100%***********************]  1 of 1 completed

Loaded: (2514, 6)
Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']


,Date,Close,High,Low,Open,Volume
0,2016-04-13,42.493156,42.572765,41.449403,41.705919,10882500
1,2016-04-14,42.891205,44.049946,42.696606,43.298091,15906500
2,2016-04-15,42.015511,43.174255,41.705924,43.112338,11822200
3,2016-04-18,41.175205,42.148198,40.874462,42.148198,11664700
4,2016-04-19,41.458252,41.944747,41.219427,41.299035,8759000


## 12 · Data Validation Checks

In [5]:
print('Missing:', df.isnull().sum().sum())
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
print(f'Rows: {len(df)}')

Missing: 0
Date range: 2016-04-13 00:00:00 to 2026-04-13 00:00:00
Rows: 2514


## 13 · Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(df['Date'], df[TARGET])
axes[0,0].set_title('DAL Close Price Over Time')

df['Volume'].plot(ax=axes[0,1], alpha=0.5)
axes[0,1].set_title('Trading Volume')

df[TARGET].hist(bins=50, ax=axes[1,0], edgecolor='black')
axes[1,0].set_title('Close Price Distribution')

df[TARGET].pct_change().hist(bins=50, ax=axes[1,1], edgecolor='black')
axes[1,1].set_title('Daily Returns')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [7]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

count    2514.000000
mean       44.421727
std        10.205458
min        18.618202
25%        37.258952
50%        44.325018
75%        50.949889
max        75.145821
Name: Close, dtype: float64

Skewness: 0.27


## 15 · Train / Validation / Test Split

Random split for this educational exercise. A time-based split would be more appropriate for production.

## 16 · Preprocessing / 17 · Feature Engineering

We engineer lag, rolling, and date-based features from the price data.

In [8]:
df = df.sort_values('Date').reset_index(drop=True)

# Date features
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['dayofweek'] = df['Date'].dt.dayofweek
df['dayofyear'] = df['Date'].dt.dayofyear

# Lag features
for lag in [1, 3, 5, 10, 20]:
    df[f'close_lag_{lag}'] = df[TARGET].shift(lag)

# Rolling features
for w in [5, 10, 20, 50]:
    df[f'roll_mean_{w}'] = df[TARGET].shift(1).rolling(w).mean()
    df[f'roll_std_{w}'] = df[TARGET].shift(1).rolling(w).std()

# Returns
df['return_1d'] = df[TARGET].pct_change(1)
df['return_5d'] = df[TARGET].pct_change(5)

# Volume features
df['vol_roll_mean_20'] = df['Volume'].shift(1).rolling(20).mean()

df = df.drop(columns=['Date']).dropna().reset_index(drop=True)

feature_cols = [c for c in df.columns if c != TARGET]
X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (1971, 24), Test: (493, 24)


## 18 · Baseline Model

In [9]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

Baseline LR: RMSE=0.24, MAE=0.16, R²=0.9994


## 19 · LazyPredict Benchmark

In [10]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

                            Adjusted R-Squared  R-Squared      RMSE  \
Model                                                                 
HuberRegressor                        0.999458   0.999485  0.223531   
LinearSVR                             0.999452   0.999479  0.224698   
PassiveAggressiveRegressor            0.999436   0.999464  0.228029   
LarsCV                                0.999421   0.999449  0.231021   
LassoLarsCV                           0.999404   0.999433  0.234372   
RidgeCV                               0.999402   0.999431  0.234769   
BayesianRidge                         0.999398   0.999427  0.235639   
LassoLarsIC                           0.999397   0.999427  0.235766   
RANSACRegressor                       0.999397   0.999426  0.235803   
LinearRegression                      0.999396   0.999426  0.235954   
TransformedTargetRegressor            0.999396   0.999426  0.235954   
Lars                                  0.999394   0.999424  0.236344   
Ridge 

## 20 · FLAML AutoML Run

In [11]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

FLAML failed: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## 21 · Boosting Models

In [12]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

CatBoost: RMSE=0.47, MAE=0.34, R²=0.9977
LightGBM: RMSE=0.39, MAE=0.28, R²=0.9984


XGBoost: RMSE=0.39, MAE=0.27, R²=0.9984


## 22 · Top 3 Model Selection

In [13]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

All models ranked by RMSE:
  Baseline_LR           RMSE=0.24  MAE=0.16  R²=0.9994
  XGBoost               RMSE=0.39  MAE=0.27  R²=0.9984
  LightGBM              RMSE=0.39  MAE=0.28  R²=0.9984
  CatBoost              RMSE=0.47  MAE=0.34  R²=0.9977

Top 3: ['Baseline_LR', 'XGBoost', 'LightGBM']


## 23 · Final Eval of Top 3

In [14]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

Top 3 Final Results:
Baseline_LR: RMSE=0.24, MAE=0.16, R²=0.9994
XGBoost: RMSE=0.39, MAE=0.27, R²=0.9984
LightGBM: RMSE=0.39, MAE=0.28, R²=0.9984


## 24 · Error Analysis

In [15]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models['CatBoost']
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **Lag features** dominate — yesterday's price predicts today's.
- This is essentially modeling momentum and mean reversion.
- **CAUTION**: High R² on random splits is misleading for stock prediction.
- This model should NOT be used for trading decisions.

## 26 · Limitations

- Random split violates temporal ordering.
- No fundamental data (earnings, revenue, economic indicators).
- Stock prices are notoriously hard to predict.
- Past performance doesn't predict future results.

## 27 · How to Improve

1. Use walk-forward validation.
2. Add fundamental data.
3. Include market-wide features (S&P 500).
4. Add sentiment data.
5. Try sequence models (LSTM).

## 28 · Production Considerations

- Financial prediction models require rigorous backtesting.
- Regulatory requirements for investment advice.
- Model drift is extreme in financial markets.
- Never deploy without comprehensive risk management.

## 29 · Common Mistakes

1. Using random splits for time-series data.
2. Claiming predictive ability from high R².
3. Not using shift() for lag features (look-ahead bias).
4. Ignoring transaction costs in profit calculations.

## 30 · Mini Challenge

1. Implement walk-forward validation.
2. Add RSI and Bollinger Band indicators.
3. Predict returns instead of prices.
4. Compare with a naive 'yesterday price' baseline.

## 31 · Final Summary

- Feature engineering from price data creates many predictive features.
- High R² on random splits is misleading for financial data.
- This project teaches financial ML fundamentals and responsible practices.
- Never confuse in-sample fit with out-of-sample predictive power.

In [16]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')

Metrics saved to metrics.json

Notebook complete.
